In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlproyectomarlon")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/everyday_Sales.csv"

In [0]:
df_sales = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
sales_schema = StructType(fields=[StructField("Date", DateType(), False),
                                     StructField("Time", StringType(), True),
                                     StructField("Item Code", StringType(), True),
                                     StructField("Quantity Sold (kilo)", DoubleType(), True),
                                     StructField("Unit Selling Price (RMB/kg)", DoubleType(), True),
                                     StructField("Sale or Return", StringType(), True),
                                     StructField("Discount (Yes/No)", StringType(), True)
])

In [0]:
df_sales_final = spark.read\
.option('header', True)\
.schema(sales_schema)\
.csv(ruta)

In [0]:
sales_selected_df = df_sales_final.select(col("Date"), 
                                                col("Time"), 
                                                col("Item Code"), 
                                                col("Quantity Sold (kilo)"), 
                                                col("Unit Selling Price (RMB/kg)"), 
                                                col("Sale or Return"),
                                                col("Discount (Yes/No)"))

In [0]:
sales_renamed_df = sales_selected_df.withColumnRenamed("Date", "sale_date") \
                                            .withColumnRenamed("Time", "sale_time") \
                                            .withColumnRenamed("Item Code", "item_code") \
                                            .withColumnRenamed("Quantity Sold (kilo)", "quantity_sold_kg") \
                                            .withColumnRenamed("Unit Selling Price (RMB/kg)", "unit_price_rmb_per_kg") \
                                            .withColumnRenamed("Sale or Return", "transaction_type") \
                                            .withColumnRenamed("Discount (Yes/No)", "has_discount")     

In [0]:
sales_final_df = sales_renamed_df

In [0]:
sales_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.everyday_Sales")

In [0]:
sales_final_df.show(10, truncate=False)